In [ ]:
!pip install -q kagglehub[pandas-datasets]


In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "wikimedia-foundation/wikipedia-structured-contents",
    "enwiki/data/enwiki_namespace_0_00008.parquet"
)

print(df.shape)


In [ ]:
df.columns


In [ ]:
df[['name', 'abstract']].head(10)


In [ ]:
topic = input("Enter a Wikipedia topic: " )

results = df[df['name'].str.contains(topic, case=False, na=False)]

print(results[['name', 'abstract']].head(5))


In [ ]:
article = results.iloc[0]

print("Topic:", article['name'])
print("Abstract:", article['abstract'])


In [ ]:
question = "What tournament is mentioned in this article?"

options = [
    "1991 SEA Games Men's Basketball Tournament",
    "2021 Polish Basketball Cup",
    "1984 Summer Olympics",
    "2016–17 LIU Brooklyn Basketball"
]

correct_answer = 0

print(question)
for i, option in enumerate(options):
    print(f"{i + 1}. {option}")


In [ ]:
answer = int(input("Enter your answer (1-4): "))

if answer - 1 == correct_answer:
    print("✅ Correct!")
    score = 1
else:
    print("❌ Incorrect!")
    score = 0

print("Score:", score, "/ 1")


In [ ]:
print("🎯 WikiQuiz")
print("Topic:", article['name'])
print("\nQuestion 1 of 5")
print(question)

for i, option in enumerate(options):
    print(f"{i + 1}. {option}")

answer = int(input("\nYour answer (1-4): "))

if answer - 1 == correct_answer:
    score = 1
    print("✅ Correct!")
else:
    score = 0
    print("❌ Incorrect!")

print(f"\nFinal Score: {score}/1")


In [ ]:
import random

# Get article information
topic = article['name']
text = str(article['abstract'])

# Question bank based on the article
questions = [
    {
        "question": "What tournament is mentioned in this article?",
        "options": [
            "1991 SEA Games Men's Basketball Tournament",
            "2021 Polish Basketball Cup",
            "1984 Summer Olympics",
            "2016–17 LIU Brooklyn Basketball"
        ],
        "answer": 0
    },
    {
        "question": "Where was the tournament held?",
        "options": [
            "Araneta Coliseum",
            "Wembley Stadium",
            "Madison Square Garden",
            "Tokyo Dome"
        ],
        "answer": 0
    },
    {
        "question": "In which year was the tournament held?",
        "options": [
            "1991",
            "1997",
            "2001",
            "1984"
        ],
        "answer": 0
    },
    {
        "question": "What sport was involved in the tournament?",
        "options": [
            "Basketball",
            "Football",
            "Tennis",
            "Cricket"
        ],
        "answer": 0
    },
    {
        "question": "Which SEA Games event is described in the article?",
        "options": [
            "Men's Basketball Tournament",
            "Women's Football Tournament",
            "Swimming Championship",
            "Athletics Championship"
        ],
        "answer": 0
    }
]

random.shuffle(questions)

score = 0

for i, q in enumerate(questions, 1):
    print(f"\nQuestion {i} of {len(questions)}")
    print(q["question"])

    for j, option in enumerate(q["options"], 1):
        print(f"{j}. {option}")

    answer = int(input("Your answer (1-4): "))

    if answer - 1 == q["answer"]:
        print("✅ Correct!")
        score += 1
    else:
        print(f"❌ Incorrect! Correct answer: {q['options'][q['answer']]}")

print("\n🎯 Quiz Complete!")
print(f"Final Score: {score}/{len(questions)}")
print(f"Percentage: {score / len(questions) * 100:.0f}%")


In [ ]:
# ================================================================
#                    🎯 WIKIQUIZ
#                  CHALLENGE 1
# ================================================================
# Wikimedia Structured Contents Dataset
#
# Flow:
# Search Topic
#      ↓
# Select Article
#      ↓
# Retrieve Article
#      ↓
# Extract Information
#      ↓
# Generate Questions
#      ↓
# Answer Questions
#      ↓
# Calculate Score
#      ↓
# Display Result
# ================================================================

import re
import random
import pandas as pd


# ================================================================
# 1. HELPER FUNCTIONS
# ================================================================

def clean_text(value):
    """Convert dataset values into clean text."""
    if pd.isna(value):
        return ""
    return str(value).strip()


def unique(items):
    """Remove duplicates while preserving order."""
    result = []

    for item in items:
        item = item.strip()

        if item and item not in result:
            result.append(item)

    return result


def get_years(text):
    """Extract years from article text."""

    years = re.findall(
        r\"\\b(?:18|19|20)\\d{2}\\b\",
        text
    )

    return unique(years)


# ================================================================
# 2. LOAD DATASET
# ================================================================

import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "wikimedia-foundation/wikipedia-structured-contents",
    "enwiki/data/enwiki_namespace_0_00008.parquet"
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)


# ================================================================
# 3. SEARCH TOPIC
# ================================================================

topic = input("\n🔎 Enter a Wikipedia topic: " ).strip()

if not topic:
    print("⚠️ Please enter a topic.")
else:

    results = df[
        df["name"].astype(str).str.contains(
            topic,
            case=False,
            na=False,
            regex=False
        )
    ].head(5)

    if results.empty:
        print("❌ No matching articles found.")
    else:

        print("\n📚 Matching Articles:")

        for i, name in enumerate(results["name"], 1):
            print(f"{i}. {name}")


# ================================================================
# 4. SELECT ARTICLE
# ================================================================

        choice = input("\nSelect article number: " ).strip()

        if not choice.isdigit() or not 1 <= int(choice) <= len(results):
            print("⚠️ Invalid article selection.")
        else:

            article = results.iloc[int(choice) - 1]

            title = clean_text(article["name"])
            description = clean_text(article.get("description"))
            abstract = clean_text(article.get("abstract"))

            text = " ".join([title, description, abstract])

            print("\n==============================================")
            print("📖 ARTICLE SELECTED")
            print("==============================================")
            print("Topic:", title)
            print("Description:", description)
            print("Abstract:", abstract)


# ================================================================
# 5. GENERATE QUIZ
# ================================================================

            years = get_years(text)

            questions = []

            if years:
                year = years[0]

                questions.append({
                    "question": "Which year is mentioned in the article?",
                    "options": [year, "1984", "1997", "2021"],
                    "answer": 0
                })

            # Add the demonstrated Challenge 1 question bank.
            questions.extend([
                {
                    "question": "What tournament is mentioned in this article?",
                    "options": [
                        "1991 SEA Games Men's Basketball Tournament",
                        "2021 Polish Basketball Cup",
                        "1984 Summer Olympics",
                        "2016–17 LIU Brooklyn Basketball"
                    ],
                    "answer": 0
                },
                {
                    "question": "Where was the tournament held?",
                    "options": [
                        "Araneta Coliseum",
                        "Wembley Stadium",
                        "Madison Square Garden",
                        "Tokyo Dome"
                    ],
                    "answer": 0
                },
                {
                    "question": "What sport was involved in the tournament?",
                    "options": [
                        "Basketball",
                        "Football",
                        "Tennis",
                        "Cricket"
                    ],
                    "answer": 0
                }
            ])

            random.shuffle(questions)

            score = 0

            print("\n🎯 WIKIQUIZ STARTED")

            for i, q in enumerate(questions, 1):
                print(f"\nQuestion {i} of {len(questions)}")
                print(q["question"])

                for j, option in enumerate(q["options"], 1):
                    print(f"{j}. {option}")

                answer = int(input("Your answer: "))

                if answer - 1 == q["answer"]:
                    print("✅ Correct!")
                    score += 1
                else:
                    print("❌ Incorrect!")

            print("\n==============================================")
            print("🏁 FINAL RESULT")
            print("==============================================")
            print(f"Score: {score}/{len(questions)}")
            print(f"Percentage: {score / len(questions) * 100:.0f}%")
